# Week 5, Lab 4 — Same task, two frameworks


In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

print("AutoGen AgentChat installed successfully!")
print("AutoGen Ext installed successfully!")

AutoGen AgentChat installed successfully!
AutoGen Ext installed successfully!


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [3]:
import zipfile
import os

zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


In [4]:
WEEK = 'Week 5'
LAB = 'Lab 4 — comparison'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 5 / Lab 4 — comparison
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [5]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn pyautogen pydantic-ai openai
else:
    %pip install -q pyautogen pydantic-ai ollama openai


In [8]:
!pip install -q transformers accelerate fastapi uvicorn

from transformers import pipeline
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn, threading

print("Loading Qwen model...")

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
)

app = FastAPI()

class ChatRequest(BaseModel):
    model: str
    messages: list
    temperature: float = 0.2

@app.post("/v1/chat/completions")
def chat(req: ChatRequest):
    prompt = req.messages[-1]["content"]

    response = pipe(
        prompt,
        max_new_tokens=200,
        temperature=req.temperature,
        do_sample=True,
    )[0]["generated_text"]

    return {
        "id": "chatcmpl-local",
        "object": "chat.completion",
        "choices": [{
            "index": 0,
            "message": {
                "role": "assistant",
                "content": response
            },
            "finish_reason": "stop"
        }]
    }

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8765)

threading.Thread(target=run_server, daemon=True).start()

print("✅ Server running at http://127.0.0.1:8765/v1")

Loading Qwen model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ Server running at http://127.0.0.1:8765/v1


In [9]:
import requests

response = requests.post(
    "http://127.0.0.1:8765/v1/chat/completions",
    json={
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
        "messages": [{"role": "user", "content": "Say hello"}]
    },
)

print(response.status_code)
print(response.json())

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


INFO:     127.0.0.1:33760 - "POST /v1/chat/completions HTTP/1.1" 200 OK
200
{'id': 'chatcmpl-local', 'object': 'chat.completion', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "Say hello to the new year with a fresh start and a new chapter in your life. Here are some tips for making the most of this time:\n\n1. Set goals: Start by setting clear, achievable goals for yourself. This will help you stay focused and motivated throughout the new year.\n\n2. Take care of yourself: Make sure you're taking care of yourself physically, mentally, and emotionally. This includes getting enough sleep, eating well, exercising regularly, and practicing self-care activities like meditation or yoga.\n\n3. Plan ahead: Before the new year starts, plan out what you want to accomplish. This can include setting up a schedule for work, school, and other responsibilities, as well as planning for fun activities and events.\n\n4. Stay positive: Keep a positive attitude and focus on the goo

In [10]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Local OpenAI-compatible Qwen server
model_client = OpenAIChatCompletionClient(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    base_url="http://127.0.0.1:8765/v1",
    api_key="EMPTY",
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "structured_output": False,
        "family": "qwen",
    },
)

# Python tools
def calc(expression: str) -> str:
    return calculator(expression)

def fact(topic: str) -> str:
    return lookup_fact(topic)

assistant = AssistantAgent(
    name="assistant",
    model_client=model_client,
    tools=[calc, fact],
    system_message=(
        "Use calc for arithmetic and fact for knowledge lookup. "
        "Answer after using the appropriate tool."
    ),
)

result = await assistant.run(
    task="What is 45*12+30? Use the calculator tool."
)

print(result.messages[-1].content)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     127.0.0.1:36302 - "POST /v1/chat/completions HTTP/1.1" 200 OK
What is 45*12+30? Use the calculator tool. To solve the expression \( 45 \times 12 + 30 \), you can use a calculator or perform it step-by-step as follows:

First, multiply 45 by 12:
\[ 45 \times 12 = 540 \]

Then add 30 to the result:
\[ 540 + 30 = 570 \]

So, \( 45 \times 12 + 30 = 570 \). The final answer is 570. If you need to verify this using a calculator, you would input \( 45 \times 12 + 30 \) into your calculator and get the same result of 570.


/usr/local/lib/python3.13/dist-packages/autogen_agentchat/agents/_assistant_agent.py:1109: UserWarning: Resolved model mismatch: Qwen/Qwen2.5-0.5B-Instruct != None. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to None to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


In [2]:
!pip install -q fastapi uvicorn transformers accelerate

from transformers import pipeline
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import threading
import time

print("Loading Qwen model... (takes 30-60 seconds first time)")

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto"
)

app = FastAPI()

class ChatRequest(BaseModel):
    model: str
    messages: list
    temperature: float = 0.2

@app.post("/v1/chat/completions")
def chat(req: ChatRequest):
    prompt = req.messages[-1]["content"]

    output = pipe(
        prompt,
        max_new_tokens=200,
        temperature=req.temperature,
        do_sample=True,
        clean_up_tokenization_spaces=False,
    )[0]["generated_text"]

    if output.startswith(prompt):
        output = output[len(prompt):].strip()

    return {
        "id": f"chatcmpl-{int(time.time())}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{
            "index": 0,
            "message": {
                "role": "assistant",
                "content": output
            },
            "finish_reason": "stop"
        }],
        "usage": {
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0
        }
    }

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8765)

threading.Thread(target=run_server, daemon=True).start()

print("✅ Server started at http://127.0.0.1:8765/v1")

Loading Qwen model... (takes 30-60 seconds first time)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Server started at http://127.0.0.1:8765/v1


In [6]:
import requests

response = requests.post(
    "http://127.0.0.1:8765/v1/chat/completions",
    json={
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
        "messages": [
            {"role": "user", "content": "Say hello"}
        ]
    }
)

print(response.status_code)
print(response.json())

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     127.0.0.1:49436 - "POST /v1/chat/completions HTTP/1.1" 200 OK
200
{'id': 'chatcmpl-1789224492', 'object': 'chat.completion', 'created': 1789224492, 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "to the new year with a fresh start and a new chapter in your life. Here are some tips for making the most of this time:\n\n1. Set goals: Start by setting clear, achievable goals for yourself. This could be anything from improving your health or learning a new skill.\n\n2. Take care of yourself: Make sure you're taking care of yourself physically, mentally, and emotionally. This includes getting enough sleep, eating healthy foods, exercising regularly, and practicing mindfulness or meditation.\n\n3. Stay connected: Connect with friends and family members who support you. Consider joining a club or group that shares your interests.\n\n4. Learn something new: Try something new each day, whether it's reading a book, watching a 

In [7]:
!pip install -q pydantic-ai openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.9/103.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.7/859.7 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.8/118.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.8/474.

In [12]:
import json
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

# ---------------- Local Qwen ----------------

model = OpenAIChatModel(
    "Qwen/Qwen2.5-0.5B-Instruct",
    provider=OpenAIProvider(
        base_url="http://127.0.0.1:8765/v1",
        api_key="EMPTY",
    ),
)

# ---------------- Schema ----------------

class Route(BaseModel):
    kind: str
    expression: str | None = None
    topic: str | None = None

# ---------------- Router ----------------

router = Agent(
    model=model,
    instructions="""
Classify the user's question.

Answer with ONLY one word:
- math
- research

No explanation.
"""
)

# ---------------- Local Tools ----------------

def calculator(expression):
    return str(eval(expression))

def lookup_fact(topic):
    facts = {
        "langgraph": "LangGraph builds stateful LLM workflows as graphs of nodes and edges.",
        "ollama": "Ollama lets you run open-source LLMs locally on your computer.",
        "mcp": "Model Context Protocol connects AI models with external tools and data."
    }
    return facts.get(topic.lower(), "No fact found.")

# ---------------- Run ----------------

question = "What is LangGraph?"

result = await router.run(question)

label = str(result.output).strip().lower()

print("Model classified as:", label)

# Build Route object in Python
if "math" in label:
    route = Route(kind="math", expression="45*12+30")
else:
    route = Route(kind="research", topic="LangGraph")

print(route)

# Call tool
if route.kind == "math":
    print("Tool Output:", calculator(route.expression))
else:
    print("Tool Output:", lookup_fact(route.topic))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     127.0.0.1:46228 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Model classified as: what are its main features and functionalities?

i am trying to understand the concept of a graph, but i'm having trouble understanding what langgraph is. it seems like it's not related to any standard library or package that i can find online.

the documentation for langgraph does not provide much information about its purpose or functionality. the only thing i found was this:

```
import graph
graph = graph()
```

this appears to be an example of how to create a new `graph` object in python using the `graph` class from the `langgraph` module. however, i'm still unsure of what langgraph is or what it does.

can anyone help me understand what langgraph is and what its main features and functionalities are?
langgraph is a python package designed specifically for working with graphs. it provides a high-level api for creating, manipulating, and analyzing graphs. 

one of the key features of langgr

## Fill this table

| | Week 1 loop | LangGraph | AutoGen | Pydantic AI | CrewAI |
|---|---|---|---|---|---|
| Control flow | | | | | |
| Tools | | | | | |
| Best when | | | | | |
